# Exact DMD with NumPy

Goal: implement Exact Dynamic Mode Decomposition without calling a DMD package.

The workflow is:

1. Build time-ordered snapshots.
2. Construct `X` and `X_prime`.
3. Compute a truncated SVD of `X`.
4. Build the reduced operator `A_tilde`.
5. Compute DMD eigenvalues and modes.
6. Reconstruct and forecast the dynamics.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

from dmd import exact_dmd, make_synthetic_flow, reconstruct, relative_error, snapshot_pair

plt.rcParams["figure.dpi"] = 130


## 1. Generate a simple flow-like dataset

Each column of `data` is one spatial snapshot. This synthetic field is not a CFD solution; it is a controlled test case with two decaying oscillatory spatial structures. That makes it a good first DMD sanity check.

In [ ]:
x, t, data = make_synthetic_flow(nx=220, nt=180)

print("x shape:", x.shape)
print("t shape:", t.shape)
print("data shape:", data.shape)  # (space, time)

fig, ax = plt.subplots(figsize=(7.0, 3.2))
im = ax.imshow(
    data,
    aspect="auto",
    origin="lower",
    extent=[t[0], t[-1], x[0], x[-1]],
    cmap="RdBu_r",
)
ax.set_title("Synthetic flow-like snapshots")
ax.set_xlabel("t")
ax.set_ylabel("x")
fig.colorbar(im, ax=ax, label="u(x, t)")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_synthetic_data.png", bbox_inches="tight")
plt.show()


In [ ]:
from dmd import make_lorenz, snapshot_pair, exact_dmd, reconstruct, relative_error

t, data = make_lorenz(nt=5000, dt=0.01)

X, Xp = snapshot_pair(data)

result = exact_dmd(X, Xp, rank=3)

data_rec = reconstruct(result, steps=data.shape[1], x0=data[:, 0])

print(relative_error(data, data_rec))

## 2. Build snapshot matrices

DMD starts from paired snapshots:

$$X = [x_1, x_2, \dots, x_{m-1}], \quad X' = [x_2, x_3, \dots, x_m].$$

The method approximates a linear time-advance operator:

$$X' \approx A X.$$

In [ ]:
X, Xp = snapshot_pair(data)

print("X shape: ", X.shape)
print("Xp shape:", Xp.shape)


## 3. Fit Exact DMD

The implementation in `src/dmd.py` follows:

$$X = U \Sigma V^*,$$

$$\tilde A = U_r^* X' V_r \Sigma_r^{-1},$$

$$\tilde A W = W \Lambda,$$

$$\Phi = X' V_r \Sigma_r^{-1} W.$$

`eigvals` are the diagonal entries of $\Lambda$, and `modes` are the columns of $\Phi$.

In [ ]:
rank = 4
result = exact_dmd(X, Xp, rank=rank)

print("rank:", result.rank)
print("eigvals:")
for i, lam in enumerate(result.eigvals, start=1):
    print(f"  {i}: {lam.real:+.5f} {lam.imag:+.5f}j |lambda|={abs(lam):.5f}")
print("modes shape:", result.modes.shape)


## 4. Plot DMD eigenvalues

Interpretation:

- Eigenvalues near the unit circle correspond to persistent dynamics.
- Eigenvalues inside the unit circle decay.
- The angle gives the oscillation frequency in discrete time.

In [ ]:
theta = np.linspace(0.0, 2.0 * np.pi, 400)

fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.plot(np.cos(theta), np.sin(theta), "k--", linewidth=1.0, label="unit circle")
ax.scatter(result.eigvals.real, result.eigvals.imag, s=55, c="crimson", zorder=3)
ax.axhline(0.0, color="0.7", linewidth=0.8)
ax.axvline(0.0, color="0.7", linewidth=0.8)
ax.set_xlabel("Real")
ax.set_ylabel("Imaginary")
ax.set_title("DMD eigenvalues")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_dmd_eigenvalues.png", bbox_inches="tight")
plt.show()


## 5. Plot DMD modes

Each DMD mode is a spatial pattern. Since modes are generally complex-valued, first inspect their real parts and magnitudes.

In [ ]:
fig, axes = plt.subplots(rank, 2, figsize=(8.0, 1.8 * rank), sharex=True)

for i in range(rank):
    axes[i, 0].plot(x, result.modes[:, i].real, color="navy")
    axes[i, 0].set_ylabel(f"mode {i + 1}")
    axes[i, 0].set_title("real part" if i == 0 else "")

    axes[i, 1].plot(x, np.abs(result.modes[:, i]), color="darkorange")
    axes[i, 1].set_title("magnitude" if i == 0 else "")

axes[-1, 0].set_xlabel("x")
axes[-1, 1].set_xlabel("x")
fig.suptitle("DMD modes", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_dmd_modes.png", bbox_inches="tight")
plt.show()


In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot(data[0], data[1], data[2])
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
plt.show()

## 6. Reconstruct the training trajectory

DMD represents the field as a sum of modes with exponential time dynamics:

$$x_k \approx \Phi \Lambda^k b.$$

In [ ]:
data_rec = reconstruct(result, steps=data.shape[1], x0=data[:, 0])
err = relative_error(data, data_rec)
print(f"relative reconstruction error: {err:.4e}")

fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.0), sharey=True)

items = [
    (data, "Original"),
    (data_rec, "DMD reconstruction"),
    (np.abs(data - data_rec), "Absolute error"),
]

for ax, (field, title) in zip(axes, items):
    im = ax.imshow(
        field,
        aspect="auto",
        origin="lower",
        extent=[t[0], t[-1], x[0], x[-1]],
        cmap="RdBu_r" if title != "Absolute error" else "magma",
    )
    ax.set_title(title)
    ax.set_xlabel("t")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

axes[0].set_ylabel("x")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_dmd_reconstruction.png", bbox_inches="tight")
plt.show()


## 7. Short-term prediction

Fit DMD only on early snapshots, then extrapolate forward. This is closer to a forecasting task than a full-trajectory reconstruction task.

In [ ]:
train_nt = 110

X_train = data[:, : train_nt - 1]
Xp_train = data[:, 1:train_nt]

pred_result = exact_dmd(X_train, Xp_train, rank=rank)
data_pred = reconstruct(pred_result, steps=data.shape[1], x0=data[:, 0])

train_error = relative_error(data[:, :train_nt], data_pred[:, :train_nt])
test_error = relative_error(data[:, train_nt:], data_pred[:, train_nt:])

print(f"train relative error: {train_error:.4e}")
print(f"test relative error:  {test_error:.4e}")

probe_idx = len(x) // 2

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.2))

axes[0].plot(t, data[probe_idx, :], label="true", linewidth=2.0)
axes[0].plot(t, data_pred[probe_idx, :], "--", label="DMD", linewidth=2.0)
axes[0].axvline(t[train_nt], color="k", linestyle=":", label="train/test split")
axes[0].set_title(f"Prediction at x={x[probe_idx]:.2f}")
axes[0].set_xlabel("t")
axes[0].set_ylabel("u")
axes[0].legend(loc="best")

time_error = np.linalg.norm(data - data_pred, axis=0) / np.linalg.norm(data, axis=0)
axes[1].plot(t, time_error, color="crimson")
axes[1].axvline(t[train_nt], color="k", linestyle=":")
axes[1].set_title("Relative error by time")
axes[1].set_xlabel("t")
axes[1].set_ylabel("relative error")
axes[1].set_yscale("log")

fig.tight_layout()
fig.savefig(FIG_DIR / "05_dmd_prediction.png", bbox_inches="tight")
plt.show()


In [ ]:
plt.plot(t, data[0], label="true x")
plt.plot(t, data_rec[0], "--", label="DMD x")
plt.legend()
plt.show()

## 8. Rank sensitivity

DMD depends on the SVD truncation rank. A small rank may miss important dynamics; a large rank may fit noise or numerical artifacts.

In [ ]:
ranks = list(range(1, 11))
errors = []

for r in ranks:
    r_result = exact_dmd(X, Xp, rank=r)
    r_rec = reconstruct(r_result, steps=data.shape[1], x0=data[:, 0])
    errors.append(relative_error(data, r_rec))

fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.plot(ranks, errors, marker="o")
ax.set_xlabel("DMD rank")
ax.set_ylabel("relative reconstruction error")
ax.set_yscale("log")
ax.set_title("Rank sensitivity")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "06_rank_sensitivity.png", bbox_inches="tight")
plt.show()


## Notes to explain in your report

1. `X` and `X_prime` are paired snapshots. DMD fits a time-advance map between them.
2. SVD gives a low-rank coordinate system, so DMD does not need to form the full high-dimensional operator.
3. `A_tilde` is the projected linear evolution operator in the POD subspace.
4. DMD eigenvalues encode growth, decay, and oscillation.
5. DMD modes encode spatial structures associated with those temporal behaviors.
6. Exact DMD is a useful entry point to Koopman-based scientific machine learning.